# Alice in Wonderland - Chapter 1 Data Visualization

Visualize powerline and USB signals from HPC data location:  
`/fs/scratch/<allocation>/May29_Alice/`

**Chapter 1 Files:**
- Powerline (Input): `Chap_1_real.bin`
- USB (Output): `Chap_1_img.bin`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
import os
import glob
from IPython.display import Audio, display

# === CONFIGURATION ===
CONFIG = {
    'DATA_FOLDERS': [
        '<REPO_ROOT>/baselines/test_new_setup',
    ],
    'USE_FOLDERS': [],  # Empty = use all folders
    'SAMPLE_RATE': 200_000,  # 200 kHz
}

# Filter folders based on USE_FOLDERS setting
if CONFIG['USE_FOLDERS']:
    filtered_folders = []
    for folder in CONFIG['DATA_FOLDERS']:
        folder_name = os.path.basename(folder)
        if folder_name in CONFIG['USE_FOLDERS']:
            filtered_folders.append(folder)
    CONFIG['DATA_FOLDERS'] = filtered_folders
    print(f"📁 Using selected folders: {CONFIG['USE_FOLDERS']}")
else:
    print(f"📁 Using all available folders")

print("\nAvailable Data Folders:")
for folder in CONFIG['DATA_FOLDERS']:
    print(f"  - {folder}")

# For single-file visualization, use the first folder
DATA_DIR = CONFIG['DATA_FOLDERS'][0]
SAMPLE_RATE = CONFIG['SAMPLE_RATE']

# Find all available files in the selected folder
available_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_real.bin')))
if len(available_files) == 0:
    print(f"\n⚠️  No files found in {DATA_DIR}")
    powerline_path = None
    usb_path = None
else:
    # Use the first file by default (you can change the index to select different files)
    FILE_INDEX = 0  # Change this to select different files (0 = first file)
    
    selected_file = available_files[FILE_INDEX]
    powerline_path = selected_file
    usb_path = selected_file.replace('_real.bin', '_img.bin')
    
    print(f"\n🔍 Single File Visualization:")
    print(f"  Folder: {DATA_DIR}")
    print(f"  Available files: {len(available_files)}")
    print(f"  Selected file index: {FILE_INDEX}")
    print(f"  Selected file: {os.path.basename(powerline_path)}")
    print(f"  Powerline: {powerline_path}")
    print(f"  USB: {usb_path}")
    
    # For legacy compatibility with cells that use CHAPTER variable
    CHAPTER = os.path.basename(powerline_path).replace('_real.bin', '')

## Data Folder Preview

Preview available data files in each configured folder.

In [ ]:
# Preview data files in each folder
print("=" * 70)
print("DATA FOLDER PREVIEW")
print("=" * 70)

for folder in CONFIG['DATA_FOLDERS']:
    folder_name = os.path.basename(folder)
    powerline_files = sorted(glob.glob(os.path.join(folder, '*_real.bin')))
    
    print(f"\n📁 {folder_name}: {len(powerline_files)} files")
    
    # Show first 5 files
    for i, f in enumerate(powerline_files[:5]):
        basename = os.path.basename(f).replace('_real.bin', '')
        usb_file = f.replace('_real.bin', '_img.bin')
        has_usb = '✓' if os.path.exists(usb_file) else '✗'
        print(f"  {i+1}. {has_usb} {basename}")
    
    if len(powerline_files) > 5:
        print(f"  ... and {len(powerline_files) - 5} more files")

print("\n" + "=" * 70)

## Load Data

Load the binary files containing powerline and USB signals.

In [ ]:
# === LOAD SIGNALS ===
if powerline_path is None or not os.path.exists(powerline_path):
    print("❌ No file selected or file not found. Please check the configuration.")
    print("   Make sure the selected folder contains *_real.bin files.")
else:
    powerline_data = np.fromfile(powerline_path, dtype=np.float32)
    usb_data = np.fromfile(usb_path, dtype=np.float32)

    print(f"\n📊 Data Statistics:")
    print(f"Powerline samples: {len(powerline_data):,}")
    print(f"USB samples: {len(usb_data):,}")
    print(f"Duration: {len(powerline_data) / SAMPLE_RATE:.2f} seconds")
    print(f"Powerline range: [{powerline_data.min():.6f}, {powerline_data.max():.6f}]")
    print(f"USB range: [{usb_data.min():.6f}, {usb_data.max():.6f}]")

## Time Domain Visualization

Plot the first 60 seconds of both signals.

In [ ]:
# === TIME DOMAIN - First 60 seconds ===
import librosa

# === FILE SELECTION FOR TIME DOMAIN PLOT ===
TIME_FILE_INDEX = 16  # Change this to select different files (0 = first file, 1 = second file, etc.)

# Get all available files from the selected folder
time_available_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_real.bin')))

if len(time_available_files) == 0:
    print(f"❌ No files found in {DATA_DIR}")
else:
    if TIME_FILE_INDEX >= len(time_available_files):
        print(f"❌ File index {TIME_FILE_INDEX} out of range. Only {len(time_available_files)} files available.")
        print(f"   Using first file instead.")
        TIME_FILE_INDEX = 0
    
    # Select file based on index
    time_powerline_path = time_available_files[TIME_FILE_INDEX]
    time_usb_path = time_powerline_path.replace('_real.bin', '_img.bin')
    file_id = os.path.basename(time_powerline_path).replace('_real.bin', '')
    
    print(f"📊 Time Domain File Selection:")
    print(f"  Available files: {len(time_available_files)}")
    print(f"  Selected index: {TIME_FILE_INDEX}")
    print(f"  Selected file: {file_id}")
    
    duration_sec = 60*1
    samples_to_plot = duration_sec * SAMPLE_RATE
    
    # Load data
    time_powerline_data = np.fromfile(time_powerline_path, dtype=np.float32, count=samples_to_plot)
    time_usb_data = np.fromfile(time_usb_path, dtype=np.float32, count=samples_to_plot)
    
    powerline_short = time_powerline_data
    usb_short = time_usb_data
    
    # Use specific MP3 file for reference (stepped tones sweep)
    mp3_path = '<REPO_ROOT>/Audio Files/stepped_tones_sweep.mp3'
    mp3_short = None
    try:
        if os.path.exists(mp3_path):
            mp3_audio, mp3_sr = librosa.load(mp3_path, sr=SAMPLE_RATE, mono=True)
            mp3_short = mp3_audio[:samples_to_plot]
            print(f"  Loaded MP3: {os.path.basename(mp3_path)}")
    except Exception as e:
        print(f"  ⚠ Could not load MP3: {e}")
    
    # Time axis (use length of powerline_short as reference)
    time_axis = np.arange(len(powerline_short)) / SAMPLE_RATE
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 11))
    
    # Powerline signal
    axes[0].plot(time_axis, powerline_short, color='orange', linewidth=0.5, alpha=0.8)
    axes[0].set_title(f'{file_id} - Powerline Signal (Input) - First {duration_sec} seconds', 
                      fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Time [seconds]')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(0, duration_sec)
    
    # USB signal
    axes[1].plot(time_axis, usb_short, color='blue', linewidth=0.5, alpha=0.8)
    axes[1].set_title(f'{file_id} - USB Signal (Output) - First {duration_sec} seconds', 
                      fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Time [seconds]')
    axes[1].set_ylabel('Amplitude')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(0, duration_sec)
    
    # MP3 (original audiobook) - if available
    if mp3_short is not None and len(mp3_short) > 0:
        axes[2].plot(time_axis[:len(mp3_short)], mp3_short, color='green', linewidth=0.5, alpha=0.8)
        axes[2].set_title(f'{file_id} - MP3 (Original) - First {duration_sec} seconds', fontsize=14, fontweight='bold')
        axes[2].set_xlabel('Time [seconds]')
        axes[2].set_ylabel('Amplitude')
        axes[2].grid(True, alpha=0.3)
        axes[2].set_xlim(0, duration_sec)
    else:
        axes[2].text(0.5, 0.5, f'MP3 not available' + (f': {os.path.basename(mp3_path)}' if mp3_path else ''), 
                     horizontalalignment='center', verticalalignment='center', transform=axes[2].transAxes)
        axes[2].set_axis_off()
    
    plt.tight_layout()
    plt.show()

In [ ]:
np.shape(mp3_audio)

## Overlay Comparison

Compare powerline and USB signals on the same plot.

In [ ]:
# === OVERLAY COMPARISON ===
plt.figure(figsize=(14, 6))
samples_overlay = min(2_000_000, len(powerline_data))
plt.plot(powerline_data[:samples_overlay], label='Powerline (Input)', color='orange', alpha=0.7, linewidth=0.5)
plt.plot(usb_data[:samples_overlay], label='USB (Output)', color='blue', alpha=0.7, linewidth=0.5)
plt.title(f"Chapter {CHAPTER} - Powerline vs USB - First {samples_overlay:,} Samples", 
          fontsize=14, fontweight='bold')
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Spectrograms

Frequency domain analysis using spectrograms.

In [ ]:
# === SPECTROGRAMS ===
import librosa

# === FILE SELECTION FOR SPECTROGRAMS ===
SPECTROGRAM_FILE_INDEX = 0  # Change this to select different files (0 = first file, 1 = second file, etc.)

# Get all available files from the selected folder
spectrogram_available_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_real.bin')))

if len(spectrogram_available_files) == 0:
    print(f"❌ No files found in {DATA_DIR}")
else:
    if SPECTROGRAM_FILE_INDEX >= len(spectrogram_available_files):
        print(f"❌ File index {SPECTROGRAM_FILE_INDEX} out of range. Only {len(spectrogram_available_files)} files available.")
        print(f"   Using first file instead.")
        SPECTROGRAM_FILE_INDEX = 0
    
    # Select file based on index
    spectrogram_powerline_path = spectrogram_available_files[SPECTROGRAM_FILE_INDEX]
    spectrogram_usb_path = spectrogram_powerline_path.replace('_real.bin', '_img.bin')
    spectrogram_file_id = os.path.basename(spectrogram_powerline_path).replace('_real.bin', '')
    
    print(f"📊 Spectrogram File Selection:")
    print(f"  Available files: {len(spectrogram_available_files)}")
    print(f"  Selected index: {SPECTROGRAM_FILE_INDEX}")
    print(f"  Selected file: {spectrogram_file_id}")
    
    spec_duration = 60
    spec_samples = spec_duration * SAMPLE_RATE
    
    # Load data
    powerline_spec = np.fromfile(spectrogram_powerline_path, dtype=np.float32, count=spec_samples)
    usb_spec = np.fromfile(spectrogram_usb_path, dtype=np.float32, count=spec_samples)
    
    # Use specific MP3 file for spectrogram reference
    mp3_path = '<REPO_ROOT>/Audio Files/stepped_tones_sweep.mp3'
    mp3_spec = None
    try:
        if os.path.exists(mp3_path):
            mp3_audio, mp3_sr = librosa.load(mp3_path, sr=SAMPLE_RATE, mono=True)
            mp3_spec = mp3_audio[:spec_samples]
            print(f"  Loaded MP3: {os.path.basename(mp3_path)}")
    except Exception as e:
        print(f"  ⚠ Could not load MP3 for spectrogram: {e}")
    
    # Compute spectrograms
    nperseg = 512 * 2
    f1, t1, Sxx1 = spectrogram(powerline_spec, fs=SAMPLE_RATE, nperseg=nperseg)
    f2, t2, Sxx2 = spectrogram(usb_spec, fs=SAMPLE_RATE, nperseg=nperseg)
    
    if mp3_spec is not None:
        f3, t3, Sxx3 = spectrogram(mp3_spec, fs=SAMPLE_RATE, nperseg=nperseg)
    
    # Enhanced contrast range in decibels
    vmin = -110
    vmax = -90
    
    # Choose layout depending on whether MP3 is available
    if mp3_spec is not None:
        fig, axs = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
        im1 = axs[0].imshow(10 * np.log10(Sxx1 + 1e-10),
                            aspect='auto', origin='lower',
                            extent=[t1.min(), t1.max(), f1.min(), f1.max()],
                            cmap='viridis', vmin=vmin, vmax=vmax)
        axs[0].set_title(f'{spectrogram_file_id} - Powerline Spectrogram (Input)', fontsize=14, fontweight='bold')
        axs[0].set_ylabel('Frequency [Hz]')
        axs[0].set_ylim(0, SAMPLE_RATE / 2)
        plt.colorbar(im1, ax=axs[0], label='Power [dB]')
    
        im2 = axs[1].imshow(10 * np.log10(Sxx2 + 1e-10),
                            aspect='auto', origin='lower',
                            extent=[t2.min(), t2.max(), f2.min(), f2.max()],
                            cmap='viridis', vmin=vmin, vmax=vmax)
        axs[1].set_title(f'{spectrogram_file_id} - USB Spectrogram (Output)', fontsize=14, fontweight='bold')
        axs[1].set_ylabel('Frequency [Hz]')
        axs[1].set_ylim(0, SAMPLE_RATE / 2)
        plt.colorbar(im2, ax=axs[1], label='Power [dB]')
    
        im3 = axs[2].imshow(10 * np.log10(Sxx3 + 1e-10),
                            aspect='auto', origin='lower',
                            extent=[t3.min(), t3.max(), f3.min(), f3.max()],
                            cmap='viridis', vmin=vmin, vmax=vmax)
        axs[2].set_title(f'{spectrogram_file_id} - MP3 Spectrogram (Original)', fontsize=14, fontweight='bold')
        axs[2].set_ylabel('Frequency [Hz]')
        axs[2].set_xlabel('Time [seconds]')
        axs[2].set_ylim(0, SAMPLE_RATE / 2)
        plt.colorbar(im3, ax=axs[2], label='Power [dB]')
    else:
        fig, axs = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
        im1 = axs[0].imshow(10 * np.log10(Sxx1 + 1e-10),
                            aspect='auto', origin='lower',
                            extent=[t1.min(), t1.max(), f1.min(), f1.max()],
                            cmap='viridis', vmin=vmin, vmax=vmax)
        axs[0].set_title(f'{spectrogram_file_id} - Powerline Spectrogram (Input)', fontsize=14, fontweight='bold')
        axs[0].set_ylabel('Frequency [Hz]')
        axs[0].set_ylim(0, SAMPLE_RATE / 2)
        plt.colorbar(im1, ax=axs[0], label='Power [dB]')
    
        im2 = axs[1].imshow(10 * np.log10(Sxx2 + 1e-10),
                            aspect='auto', origin='lower',
                            extent=[t2.min(), t2.max(), f2.min(), f2.max()],
                            cmap='viridis', vmin=vmin, vmax=vmax)
        axs[1].set_title(f'{spectrogram_file_id} - USB Spectrogram (Output)', fontsize=14, fontweight='bold')
        axs[1].set_ylabel('Frequency [Hz]')
        axs[1].set_xlabel('Time [seconds]')
        axs[1].set_ylim(0, SAMPLE_RATE / 2)
        plt.colorbar(im2, ax=axs[1], label='Power [dB]')
    
    plt.tight_layout()
    plt.show()

## Zoomed View (5 seconds)

Detailed view of the first 5 seconds.

In [ ]:
# === ZOOMED VIEW - First 5 seconds ===
zoom_sec = 5
zoom_samples = zoom_sec * SAMPLE_RATE
zoom_time = np.arange(zoom_samples) / SAMPLE_RATE

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Powerline zoomed
axes[0].plot(zoom_time, powerline_data[:zoom_samples], color='orange', linewidth=0.8)
axes[0].set_title(f'Chapter {CHAPTER} - Powerline Signal - First {zoom_sec} seconds (Zoomed)', 
                  fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time [seconds]')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, zoom_sec)

# USB zoomed
axes[1].plot(zoom_time, usb_data[:zoom_samples], color='blue', linewidth=0.8)
axes[1].set_title(f'Chapter {CHAPTER} - USB Signal - First {zoom_sec} seconds (Zoomed)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time [seconds]')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, zoom_sec)

plt.tight_layout()
plt.show()

## Audio Playback

Listen to the USB signal (downsampled for audio playback).

In [ ]:
# === AUDIO PLAYBACK ===
# Normalize USB signal for audio playback
usb_audio = usb_data / np.max(np.abs(usb_data))

print("🔊 Playing USB Signal (first 60 seconds):")
audio_samples = min(60 * SAMPLE_RATE, len(usb_audio))
display(Audio(usb_audio[:audio_samples], rate=SAMPLE_RATE))

In [ ]:
# === PSD PLOT - Power Spectral Density for playback window ===
from scipy.signal import welch

# Use the same sample window as the audio playback above
window_samples = audio_samples

# Choose a sensible nperseg for PSD (power-of-two, not greater than window)
nperseg = min(16384, window_samples)

# Compute PSDs
f_p, Pxx_p = welch(powerline_data[:window_samples], fs=SAMPLE_RATE, nperseg=nperseg)
f_u, Pxx_u = welch(usb_audio[:window_samples], fs=SAMPLE_RATE, nperseg=nperseg)

mp3_available = 'mp3_short' in globals() and mp3_short is not None and len(mp3_short) >= window_samples
if mp3_available:
    f_m, Pxx_m = welch(mp3_short[:window_samples], fs=SAMPLE_RATE, nperseg=nperseg)

# Convert to dB
Pdb_p = 10 * np.log10(Pxx_p + 1e-20)
Pdb_u = 10 * np.log10(Pxx_u + 1e-20)
if mp3_available:
    Pdb_m = 10 * np.log10(Pxx_m + 1e-20)

# Plot
if mp3_available:
    fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
else:
    fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axs[0].plot(f_p, Pdb_p, color='orange')
axs[0].set_title(f'Chapter {CHAPTER} - PSD: Powerline (first {int(window_samples/SAMPLE_RATE)} s)')
axs[0].set_ylabel('Power (dB)')
axs[0].grid(True, alpha=0.3)
axs[0].set_xlim(0, SAMPLE_RATE/2)

axs[1].plot(f_u, Pdb_u, color='blue')
axs[1].set_title(f'Chapter {CHAPTER} - PSD: USB (first {int(window_samples/SAMPLE_RATE)} s)')
axs[1].set_ylabel('Power (dB)')
axs[1].grid(True, alpha=0.3)
axs[1].set_xlim(0, SAMPLE_RATE/2)

if mp3_available:
    axs[2].plot(f_m, Pdb_m, color='green')
    axs[2].set_title(f'Chapter {CHAPTER} - PSD: MP3 (first {int(window_samples/SAMPLE_RATE)} s)')
    axs[2].set_ylabel('Power (dB)')
    axs[2].set_xlabel('Frequency [Hz]')
    axs[2].grid(True, alpha=0.3)
else:
    axs[1].set_xlabel('Frequency [Hz]')

plt.tight_layout()
plt.show()

## Powerline PSD Analysis - All Files

Compare the Power Spectral Density (PSD) of powerline signals across all files in selected folders.

In [ ]:
# === SPECTROGRAM - Selected File at 20 kHz (5 kHz bandwidth) ===

# === FILE SELECTION FOR SPECTROGRAM ===
SPEC_FILE_INDEX = 0  # Change this to select different files (0 = first file, 1 = second file, etc.)

# Get all available files from the selected folder
spec_available_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_real.bin')))

if len(spec_available_files) == 0:
    print(f"❌ No files found in {DATA_DIR}")
elif SPEC_FILE_INDEX >= len(spec_available_files):
    print(f"❌ File index {SPEC_FILE_INDEX} out of range. Only {len(spec_available_files)} files available.")
else:
    # Select file based on index
    spec_powerline_path = spec_available_files[SPEC_FILE_INDEX]
    spec_usb_path = spec_powerline_path.replace('_real.bin', '_img.bin')
    spec_chapter = os.path.basename(spec_powerline_path).replace('_real.bin', '')
    
    print(f"📊 Spectrogram File Selection:")
    print(f"  Available files: {len(spec_available_files)}")
    print(f"  Selected index: {SPEC_FILE_INDEX}")
    print(f"  Selected file: {spec_chapter}")
    print(f"Creating detailed spectrogram for: {os.path.basename(spec_powerline_path)}")
    
    # Load only the first 60 seconds of powerline and USB data
    duration_sec = 60*20
    samples_to_load = duration_sec * SAMPLE_RATE
    spec_powerline = np.fromfile(spec_powerline_path, dtype=np.float32, count=samples_to_load)
    spec_usb = np.fromfile(spec_usb_path, dtype=np.float32, count=samples_to_load)
    
    print(f"Loaded {len(spec_powerline):,} samples ({len(spec_powerline)/SAMPLE_RATE:.2f} seconds)")

In [ ]:
from scipy.signal import welch
import glob

# === PSD ANALYSIS FOR ALL FILES IN SELECTED FOLDERS ===
print("Computing PSD for all files in selected folders...")

# Collect all powerline files from all configured folders
all_powerline_files = []
for folder in CONFIG['DATA_FOLDERS']:
    folder_name = os.path.basename(folder)
    files = sorted(glob.glob(os.path.join(folder, "*_real.bin")))
    print(f"📁 {folder_name}: {len(files)} files")
    all_powerline_files.extend([(f, folder_name) for f in files])

print(f"\nTotal files to process: {len(all_powerline_files)}\n")

if len(all_powerline_files) == 0:
    print(f"No powerline files found in configured folders")
else:
    # Create figure with 2 subplots for powerline and USB
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))
    
    # Parameters for PSD computation
    nperseg = 8192  # FFT window size
    max_duration = 60.0  # Use first 60 seconds for speed
    
    # Color map for different folders
    folder_colors = {'May29_Alice': 'blue', 'July10_Podcasts': 'orange'}
    
    for file_path, folder_name in all_powerline_files:
        try:
            # Extract file identifier from filename
            filename = os.path.basename(file_path)
            file_id = filename.replace('_real.bin', '')
            
            # Load powerline and USB data (limit to first max_duration seconds)
            max_samples = int(max_duration * SAMPLE_RATE)
            powerline_data = np.fromfile(file_path, dtype=np.float32, count=max_samples)
            
            # Load corresponding USB file
            usb_file_path = file_path.replace('_real.bin', '_img.bin')
            usb_data = np.fromfile(usb_file_path, dtype=np.float32, count=max_samples)
            
            if len(powerline_data) == 0:
                print(f"⚠️  {file_id}: No data")
                continue
            
            # Compute PSD for powerline using Welch's method
            freqs_p, psd_p = welch(powerline_data, fs=SAMPLE_RATE, nperseg=nperseg, 
                                   scaling='density', return_onesided=True)
            psd_p_db = 10 * np.log10(psd_p + 1e-12)
            
            # Compute PSD for USB
            freqs_u, psd_u = welch(usb_data, fs=SAMPLE_RATE, nperseg=nperseg, 
                                   scaling='density', return_onesided=True)
            psd_u_db = 10 * np.log10(psd_u + 1e-12)
            
            # Plot with folder-specific color
            color = folder_colors.get(folder_name, 'gray')
            
            # Plot powerline PSD
            ax1.plot(freqs_p / 1000, psd_p_db, label=f'{folder_name}: {file_id}', 
                    alpha=0.6, linewidth=1.2, color=color)
            
            # Plot USB PSD
            ax2.plot(freqs_u / 1000, psd_u_db, label=f'{folder_name}: {file_id}', 
                    alpha=0.6, linewidth=1.2, color=color)
            
            print(f"✓ {folder_name}/{file_id}: {len(powerline_data):,} samples ({len(powerline_data)/SAMPLE_RATE:.1f}s)")
            
        except Exception as e:
            print(f"✗ Error processing {filename}: {e}")
    
    # Format powerline plot
    ax1.set_xlabel('Frequency (kHz)', fontsize=12)
    ax1.set_ylabel('Power Spectral Density (dB/Hz)', fontsize=12)
    ax1.set_title('Powerline Signal PSD - All Files', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax1.set_xlim(0, SAMPLE_RATE / 2000)
    
    # Format USB plot
    ax2.set_xlabel('Frequency (kHz)', fontsize=12)
    ax2.set_ylabel('Power Spectral Density (dB/Hz)', fontsize=12)
    ax2.set_title('USB Signal PSD - All Files', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax2.set_xlim(0, SAMPLE_RATE / 2000)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ PSD analysis complete for {len(all_powerline_files)} files (both powerline and USB)")

## Single File PSD Analysis

Select a specific file and analyze its Power Spectral Density in detail.

In [ ]:
# === SINGLE FILE PSD ANALYSIS ===
from scipy.signal import welch
import glob
# === FILE SELECTION ===
PSD_FILE_INDEX = 0  # Change this to select different files (0 = first file, 1 = second file, etc.)

# Get all available files from the selected folder
psd_available_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_real.bin')))

if len(psd_available_files) == 0:
    print(f"❌ No files found in {DATA_DIR}")
elif PSD_FILE_INDEX >= len(psd_available_files):
    print(f"❌ File index {PSD_FILE_INDEX} out of range. Only {len(psd_available_files)} files available.")
else:
    # Select file based on index
    psd_powerline_path = psd_available_files[PSD_FILE_INDEX]
    psd_usb_path = psd_powerline_path.replace('_real.bin', '_img.bin')
    psd_file_id = os.path.basename(psd_powerline_path).replace('_real.bin', '')
    
    print(f"📊 Single File PSD Analysis:")
    print(f"  Available files: {len(psd_available_files)}")
    print(f"  Selected index: {PSD_FILE_INDEX}")
    print(f"  Selected file: {psd_file_id}")
    
    # Load data (first 60 seconds for speed)
    max_duration = 60.0
    max_samples = int(max_duration * SAMPLE_RATE)
    psd_powerline_data = np.fromfile(psd_powerline_path, dtype=np.float32, count=max_samples)
    psd_usb_data = np.fromfile(psd_usb_path, dtype=np.float32, count=max_samples)
    
    print(f"  Loaded: {len(psd_powerline_data):,} samples ({len(psd_powerline_data)/SAMPLE_RATE:.2f} seconds)")
    
    # Compute PSD using Welch's method
    nperseg = 8192  # FFT window size
    freqs_p, psd_p = welch(psd_powerline_data, fs=SAMPLE_RATE, nperseg=nperseg, 
                           scaling='density', return_onesided=True)
    psd_p_db = 10 * np.log10(psd_p + 1e-12)
    
    freqs_u, psd_u = welch(psd_usb_data, fs=SAMPLE_RATE, nperseg=nperseg, 
                           scaling='density', return_onesided=True)
    psd_u_db = 10 * np.log10(psd_u + 1e-12)
    
    # Create plots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    
    # Plot powerline PSD
    ax1.plot(freqs_p / 1000, psd_p_db, color='blue', linewidth=1.5, alpha=0.8)
    ax1.set_xlabel('Frequency (kHz)', fontsize=12)
    ax1.set_ylabel('Power Spectral Density (dB/Hz)', fontsize=12)
    ax1.set_title(f'Powerline Signal PSD - {psd_file_id}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_xlim(0, SAMPLE_RATE / 2000)
    
    # Highlight key frequency bands
    ax1.axvspan(15, 25, alpha=0.1, color='red', label='20 kHz ±5 kHz')
    ax1.axvline(x=20, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='20 kHz center')
    ax1.legend(loc='upper right')
    
    # Plot USB PSD
    ax2.plot(freqs_u / 1000, psd_u_db, color='green', linewidth=1.5, alpha=0.8)
    ax2.set_xlabel('Frequency (kHz)', fontsize=12)
    ax2.set_ylabel('Power Spectral Density (dB/Hz)', fontsize=12)
    ax2.set_title(f'USB Signal (Output) PSD - {psd_file_id}', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.set_xlim(0, SAMPLE_RATE / 2000)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ PSD analysis complete for {psd_file_id}")
    print(f"  Frequency resolution: {freqs_p[1] - freqs_p[0]:.2f} Hz")
    print(f"  Number of frequency bins: {len(freqs_p)}")


## Selected File Spectrogram - 20 kHz Band (5 kHz bandwidth)

Detailed spectrogram of the currently selected file's powerline signal focused on 20 kHz ± 2.5 kHz.

In [ ]:

    
    # Spectrogram parameters
    nperseg = 2048  # Window size for better frequency resolution
    noverlap = nperseg // 2  # 50% overlap
    
    # Compute spectrogram
    print("Computing spectrogram...")
    f, t, Sxx = spectrogram(spec_powerline, fs=SAMPLE_RATE, nperseg=nperseg, noverlap=noverlap)
    
    # Convert to dB
    Sxx_db = 10 * np.log10(Sxx + 1e-12)

    # Focus on 20 kHz ± 5 kHz (15 - 25 kHz)
    center_freq = 20000  # 20 kHz
    bandwidth = 10000  # 10 kHz total
    freq_min = center_freq - bandwidth / 2  # 15 kHz
    freq_max = center_freq + bandwidth / 2  # 25 kHz

    # Find frequency indices
    freq_mask = (f >= freq_min) & (f <= freq_max)
    f_zoomed = f[freq_mask]
    Sxx_zoomed = Sxx_db[freq_mask, :]
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    
    # Plot 1: Powerline Spectrogram
    im = ax1.imshow(Sxx_zoomed, aspect='auto', origin='lower',
                    extent=[0, duration_sec, f_zoomed.min()/1000, f_zoomed.max()/1000],
                    cmap='gray', interpolation='bilinear')
    
    ax1.set_title(f'{spec_chapter} - Powerline Spectrogram at {center_freq/1000:.0f} kHz', 
                  fontsize=14, fontweight='bold')
    ax1.set_ylabel('Frequency (kHz)', fontsize=12)
    ax1.set_xlim(0, duration_sec)
    ax1.set_ylim(freq_min/1000, freq_max/1000)
    ax1.grid(True, alpha=0.2, color='white', linewidth=0.5)
    ax1.axhline(y=center_freq/1000, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'{center_freq/1000:.0f} kHz')
    ax1.legend(loc='upper right')
    #plt.colorbar(im, ax=ax1, label='Power (dB)')
    
    # Plot 2: USB Signal (time domain)
    time_axis = np.arange(len(spec_usb)) / SAMPLE_RATE
    ax2.plot(time_axis, spec_usb, color='blue', linewidth=0.5, alpha=0.8)
    ax2.set_title(f'{spec_chapter} - USB Signal (Output) - First {int(duration_sec)} seconds', 
                  fontsize=14, fontweight='bold')
    ax2.set_xlabel('Time (seconds)', fontsize=12)
    ax2.set_ylabel('Amplitude', fontsize=12)
    ax2.set_xlim(0, duration_sec)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Spectrogram and signal plot complete")
    print(f"  File: {os.path.basename(spec_powerline_path)}")
    print(f"  Frequency range: {freq_min/1000:.1f} - {freq_max/1000:.1f} kHz")
    print(f"  Time range: 0 - {t.max():.1f} seconds")
    print(f"  Frequency resolution: {f[1]-f[0]:.2f} Hz")


## Demodulation: Extract USB Signal from Powerline

Extract the time-domain signal encoded around 20 kHz carrier frequency by demodulating the powerline signal.

In [ ]:
# === DEMODULATION: Extract USB from Powerline ===
from scipy.signal import butter, filtfilt, hilbert, resample

if powerline_path is None or not os.path.exists(powerline_path):
    print(f"❌ No file selected. Please run the configuration and data loading cells first.")
else:
    print(f"Demodulating powerline signal: {os.path.basename(powerline_path)}")
    
    # Use the same data loaded earlier (first 60 seconds)
    duration_sec = 60
    samples_to_load = duration_sec * SAMPLE_RATE
    
    # Load fresh data for demodulation
    demod_powerline = np.fromfile(powerline_path, dtype=np.float32, count=samples_to_load)
    demod_usb = np.fromfile(usb_path, dtype=np.float32, count=samples_to_load)
    
    # Parameters
    carrier_freq = 65000  # 20 kHz carrier
    bandwidth = 10000  # 5 kHz bandwidth
    
    # Step 1: Bandpass filter around 20 kHz (17.5 - 22.5 kHz)
    print(f"Step 1: Bandpass filtering ({(carrier_freq-bandwidth/2)/1000:.1f} - {(carrier_freq+bandwidth/2)/1000:.1f} kHz)...")
    nyquist = SAMPLE_RATE / 2
    low_cutoff = (carrier_freq - bandwidth/2) / nyquist
    high_cutoff = (carrier_freq + bandwidth/2) / nyquist
    b, a = butter(4, [low_cutoff, high_cutoff], btype='band')
    filtered_signal = filtfilt(b, a, demod_powerline)
    
    # Visualize the bandpass filtered signal
    print("Visualizing bandpass filtered signal...")
    nperseg_bp = 2048
    noverlap_bp = nperseg_bp // 2
    f_bp, t_bp, Sxx_bp = spectrogram(filtered_signal, fs=SAMPLE_RATE, nperseg=nperseg_bp, noverlap=noverlap_bp)
    Sxx_bp_db = 10 * np.log10(Sxx_bp + 1e-12)
    
    # Focus on the filtered band
    freq_min_bp = carrier_freq - bandwidth / 2 - 1000  # Add 1 kHz margin for visualization
    freq_max_bp = carrier_freq + bandwidth / 2 + 1000
    freq_mask_bp = (f_bp >= freq_min_bp) & (f_bp <= freq_max_bp)
    f_bp_zoomed = f_bp[freq_mask_bp]
    Sxx_bp_zoomed = Sxx_bp_db[freq_mask_bp, :]
    
    plt.figure(figsize=(16, 6))
    im = plt.imshow(Sxx_bp_zoomed, aspect='auto', origin='lower',
                    extent=[0, duration_sec, f_bp_zoomed.min()/1000, f_bp_zoomed.max()/1000],
                    cmap='viridis', interpolation='bilinear')
    plt.colorbar(im, label='Power (dB)')
    plt.title(f'{CHAPTER} - Bandpass Filtered Signal Spectrogram (17.5-22.5 kHz)', fontsize=14, fontweight='bold')
    plt.xlabel('Time (seconds)', fontsize=12)
    plt.ylabel('Frequency (kHz)', fontsize=12)
    plt.xlim(0, duration_sec)
    plt.axhline(y=20, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='20 kHz Carrier')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.2, color='white', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    # Step 2: Frequency shift (mix down to baseband)
    print("Step 2: Frequency mixing to baseband...")
    t = np.arange(len(demod_powerline)) / SAMPLE_RATE
    # Create complex carrier for demodulation
    i_carrier = np.cos(2 * np.pi * carrier_freq * t)
    q_carrier = -np.sin(2 * np.pi * carrier_freq * t)
    
    # Mix down
    i_baseband = filtered_signal * i_carrier
    q_baseband = filtered_signal * q_carrier
    
    # Step 3: Lowpass filter both I and Q
    print("Step 3: Lowpass filtering...")
    cutoff_freq = 2500 / nyquist  # 2.5 kHz cutoff (half the bandwidth)
    b_low, a_low = butter(5, cutoff_freq, btype='low')
    i_filtered = filtfilt(b_low, a_low, i_baseband)
    q_filtered = filtfilt(b_low, a_low, q_baseband)
    
    # Step 4: Compute magnitude (envelope)
    print("Step 4: Computing envelope...")
    demodulated = np.sqrt(i_filtered**2 + q_filtered**2)
    
    # Remove DC and normalize
    demodulated = demodulated - np.mean(demodulated)
    demodulated = demodulated / np.max(np.abs(demodulated))
    
    # Compare with actual USB signal
    print("\n✓ Demodulation complete!")
    
    # Plot comparison
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    
    # Plot 1: Original powerline signal
    time_axis = np.arange(len(demod_powerline)) / SAMPLE_RATE
    axes[0].plot(time_axis, demod_powerline, color='orange', linewidth=0.5, alpha=0.8)
    axes[0].set_title(f'{CHAPTER} - Original Powerline Signal (Input)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=12)
    axes[0].set_xlim(0, duration_sec)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Demodulated signal (extracted from powerline)
    axes[1].plot(time_axis, demodulated, color='green', linewidth=0.5, alpha=0.8)
    axes[1].set_title(f'{CHAPTER} - Demodulated Signal (Extracted from Powerline)', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=12)
    axes[1].set_xlim(0, duration_sec)
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Actual USB signal (for comparison)
    axes[2].plot(time_axis, demod_usb / np.max(np.abs(demod_usb)), color='blue', linewidth=0.5, alpha=0.8)
    axes[2].set_title(f'{CHAPTER} - Actual USB Signal (Output)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Time (seconds)', fontsize=12)
    axes[2].set_ylabel('Amplitude', fontsize=12)
    axes[2].set_xlim(0, duration_sec)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate correlation
    correlation = np.corrcoef(demodulated, demod_usb / np.max(np.abs(demod_usb)))[0, 1]
    print(f"\n📊 Correlation between demodulated and actual USB: {correlation:.4f}")
    print(f"   (1.0 = perfect match, 0.0 = no correlation)")

## Bandpass Filtered Signal Analysis

Apply a bandpass filter to the selected file and visualize the filtered signal in time domain.

In [ ]:
# === BANDPASS FILTERED SIGNAL - Selected File ===
from scipy.signal import butter, filtfilt

# === FILE SELECTION FOR BANDPASS FILTER ===
BP_FILE_INDEX = 2  # Change this to select different files (0 = first file, 1 = second file, etc.)

# Get all available files from the selected folder
bp_available_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_real.bin')))

if len(bp_available_files) == 0:
    print(f"❌ No files found in {DATA_DIR}")
elif BP_FILE_INDEX >= len(bp_available_files):
    print(f"❌ File index {BP_FILE_INDEX} out of range. Only {len(bp_available_files)} files available.")
else:
    # Select file based on index
    bp_powerline_path = bp_available_files[BP_FILE_INDEX]
    bp_usb_path = bp_powerline_path.replace('_real.bin', '_img.bin')
    bp_chapter = os.path.basename(bp_powerline_path).replace('_real.bin', '')
    
    print(f"🔊 Bandpass Filter File Selection:")
    print(f"  Available files: {len(bp_available_files)}")
    print(f"  Selected index: {BP_FILE_INDEX}")
    print(f"  Selected file: {bp_chapter}")
    print(f"Applying bandpass filter to: {os.path.basename(bp_powerline_path)}")
    
    # Load data (first 60 seconds)
    duration_sec = 60
    samples_to_load = duration_sec * SAMPLE_RATE
    bp_powerline = np.fromfile(bp_powerline_path, dtype=np.float32, count=samples_to_load)
    bp_usb = np.fromfile(bp_usb_path, dtype=np.float32, count=samples_to_load)
    
    print(f"Loaded {len(bp_powerline):,} samples ({len(bp_powerline)/SAMPLE_RATE:.2f} seconds)")
    
    # Bandpass filter parameters
    center_freq = 11000  # 11 kHz center frequency
    bandwidth = 1000    # 1 kHz bandwidth
    freq_min = center_freq - bandwidth / 2  # 10 kHz
    freq_max = center_freq + bandwidth / 2  # 12 kHz

    print(f"\n🔧 Bandpass Filter Configuration:")
    print(f"  Center frequency: {center_freq/1000:.1f} kHz")
    print(f"  Bandwidth: {bandwidth/1000:.1f} kHz")
    print(f"  Passband: {freq_min/1000:.1f} - {freq_max/1000:.1f} kHz")
    
    # Design and apply bandpass filter
    print("\nApplying bandpass filter...")
    nyquist = SAMPLE_RATE / 2
    low_cutoff = freq_min / nyquist
    high_cutoff = freq_max / nyquist
    
    b, a = butter(4, [low_cutoff, high_cutoff], btype='band')
    bp_filtered = filtfilt(b, a, bp_powerline)
    
    # Create time axis
    time_axis = np.arange(len(bp_powerline)) / SAMPLE_RATE
    
    # Plot original and filtered signals
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    
    # Plot 1: Original powerline signal
    axes[0].plot(time_axis, bp_powerline, color='orange', linewidth=0.5, alpha=0.8)
    axes[0].set_title(f'{bp_chapter} - Original Powerline Signal', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=12)
    axes[0].set_xlim(0, duration_sec)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Bandpass filtered signal
    axes[1].plot(time_axis, bp_filtered, color='green', linewidth=0.5, alpha=0.8)
    axes[1].set_title(f'{bp_chapter} - Bandpass Filtered Signal ({freq_min/1000:.1f} - {freq_max/1000:.1f} kHz)', 
                      fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Amplitude', fontsize=12)
    axes[1].set_xlim(0, duration_sec)
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: USB signal for comparison
    axes[2].plot(time_axis, bp_usb, color='blue', linewidth=0.5, alpha=0.8)
    axes[2].set_title(f'{bp_chapter} - USB Signal (Output)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Time (seconds)', fontsize=12)
    axes[2].set_ylabel('Amplitude', fontsize=12)
    axes[2].set_xlim(0, duration_sec)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"\n✓ Bandpass filtering complete!")
    print(f"\n📊 Signal Statistics:")
    print(f"  Original signal - RMS: {np.sqrt(np.mean(bp_powerline**2)):.6f}")
    print(f"  Filtered signal - RMS: {np.sqrt(np.mean(bp_filtered**2)):.6f}")
    print(f"  Energy reduction: {(1 - np.mean(bp_filtered**2)/np.mean(bp_powerline**2))*100:.2f}%")
    
    # Zoomed view (first 1 second)
    zoom_samples = SAMPLE_RATE  # 1 second
    zoom_time = np.arange(zoom_samples) / SAMPLE_RATE
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    
    axes[0].plot(zoom_time, bp_powerline[:zoom_samples], color='orange', linewidth=1.0, alpha=0.8)
    axes[0].set_title(f'{bp_chapter} - Original Signal (First 1 second)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=12)
    axes[0].set_xlim(0, 1)
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(zoom_time, bp_filtered[:zoom_samples], color='green', linewidth=1.0, alpha=0.8)
    axes[1].set_title(f'{bp_chapter} - Filtered Signal (First 1 second)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Time (seconds)', fontsize=12)
    axes[1].set_ylabel('Amplitude', fontsize=12)
    axes[1].set_xlim(0, 1)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# === FREQUENCY BIN CORRELATION ANALYSIS ===
from scipy.signal import butter, filtfilt
from scipy.stats import pearsonr

print("🔍 Analyzing correlation between powerline frequency bins and USB signal...")

if powerline_path is None or not os.path.exists(powerline_path):
    print("❌ No file selected. Please run the configuration cells first.")
else:
    # Load data (first 60 seconds)
    duration_sec = 60
    samples_to_load = duration_sec * SAMPLE_RATE
    corr_powerline = np.fromfile(powerline_path, dtype=np.float32, count=samples_to_load)
    corr_usb = np.fromfile(usb_path, dtype=np.float32, count=samples_to_load)
    
    # Normalize USB signal
    corr_usb_norm = (corr_usb - np.mean(corr_usb)) / (np.std(corr_usb) + 1e-10)
    
    # Define frequency bins to test
    freq_start = 0      # 0 Hz (DC)
    freq_end = 95000    # 95 kHz (below Nyquist)
    freq_step = 1000    # 1 kHz steps
    bandwidth = 5000    # 5 kHz bandwidth for each bin
    
    frequencies = np.arange(freq_start, freq_end, freq_step)
    correlations = []
    
    print(f"\nTesting {len(frequencies)} frequency bins from {freq_start/1000:.0f} kHz to {freq_end/1000:.0f} kHz")
    print(f"Bandwidth per bin: {bandwidth/1000:.0f} kHz")
    print(f"Processing using FFT/IFFT method...")
    
    # Compute FFT of powerline signal
    fft_powerline = np.fft.fft(corr_powerline)
    freqs_fft = np.fft.fftfreq(len(corr_powerline), 1/SAMPLE_RATE)
    
    for i, center_freq in enumerate(frequencies):
        # Progress indicator
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(frequencies)} ({(i+1)/len(frequencies)*100:.1f}%)")
        
        # Define frequency range for this bin
        freq_low = center_freq - bandwidth/2
        freq_high = center_freq + bandwidth/2
        
        try:
            # Create frequency mask for this bin (both positive and negative frequencies)
            freq_mask = ((np.abs(freqs_fft) >= freq_low) & (np.abs(freqs_fft) <= freq_high))
            
            # Apply mask in frequency domain
            fft_filtered = fft_powerline.copy()
            fft_filtered[~freq_mask] = 0
            
            # Inverse FFT to get filtered signal
            filtered = np.fft.ifft(fft_filtered).real
            
            # Normalize filtered signal
            filtered_norm = (filtered - np.mean(filtered)) / (np.std(filtered) + 1e-10)
            
            # Compute correlation with normalized USB signal
            corr, _ = pearsonr(filtered_norm, corr_usb_norm)
            correlations.append(abs(corr))  # Use absolute correlation
            
        except Exception as e:
            correlations.append(0)
    
    correlations = np.array(correlations)
    
    # Find peak correlation
    max_corr_idx = np.argmax(correlations)
    max_corr_freq = frequencies[max_corr_idx]
    max_corr_value = correlations[max_corr_idx]
    
    print(f"\n✓ Analysis complete!")
    print(f"\n📊 Results:")
    print(f"  Peak correlation: {max_corr_value:.4f}")
    print(f"  Best frequency: {max_corr_freq/1000:.1f} kHz")
    print(f"  Frequency range: {(max_corr_freq-bandwidth/2)/1000:.1f} - {(max_corr_freq+bandwidth/2)/1000:.1f} kHz")
    
    # Find top 5 frequencies
    top_5_indices = np.argsort(correlations)[-5:][::-1]
    print(f"\n🏆 Top 5 Frequency Bins:")
    for rank, idx in enumerate(top_5_indices, 1):
        freq = frequencies[idx]
        corr = correlations[idx]
        print(f"  {rank}. {freq/1000:.1f} kHz: correlation = {corr:.4f}")
    
    # Plot correlation vs frequency
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    
    # Plot 1: Full spectrum
    ax1.plot(frequencies / 1000, correlations, linewidth=2, color='blue')
    ax1.axvline(x=max_corr_freq/1000, color='red', linestyle='--', linewidth=2, 
                label=f'Peak: {max_corr_freq/1000:.1f} kHz (r={max_corr_value:.4f})')
    ax1.set_xlabel('Center Frequency (kHz)', fontsize=12)
    ax1.set_ylabel('Absolute Correlation', fontsize=12)
    ax1.set_title('Correlation between Powerline Frequency Bins and USB Signal', 
                  fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=12)
    ax1.set_xlim(freq_start/1000, freq_end/1000)
    
    # Plot 2: Zoomed view around peak
    zoom_range = 10000  # ±10 kHz around peak
    zoom_mask = (frequencies >= max_corr_freq - zoom_range) & (frequencies <= max_corr_freq + zoom_range)
    zoom_freqs = frequencies[zoom_mask]
    zoom_corrs = correlations[zoom_mask]
    
    ax2.plot(zoom_freqs / 1000, zoom_corrs, linewidth=2, color='green', marker='o')
    ax2.axvline(x=max_corr_freq/1000, color='red', linestyle='--', linewidth=2,
                label=f'Peak: {max_corr_freq/1000:.1f} kHz')
    ax2.set_xlabel('Center Frequency (kHz)', fontsize=12)
    ax2.set_ylabel('Absolute Correlation', fontsize=12)
    ax2.set_title(f'Zoomed View: ±{zoom_range/1000:.0f} kHz around Peak', 
                  fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    # Visualize the best frequency bin
    print(f"\n📈 Extracting signal from best frequency bin ({max_corr_freq/1000:.1f} kHz)...")
    
    # Extract best frequency bin using FFT/IFFT
    freq_low = max_corr_freq - bandwidth/2
    freq_high = max_corr_freq + bandwidth/2
    freq_mask = ((np.abs(freqs_fft) >= freq_low) & (np.abs(freqs_fft) <= freq_high))
    
    fft_best = fft_powerline.copy()
    fft_best[~freq_mask] = 0
    best_filtered = np.fft.ifft(fft_best).real
    best_filtered_norm = (best_filtered - np.mean(best_filtered)) / (np.std(best_filtered) + 1e-10)
    
    # Plot comparison
    time_axis = np.arange(len(corr_usb)) / SAMPLE_RATE
    
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    
    # Plot 1: Original powerline
    axes[0].plot(time_axis, corr_powerline, color='orange', linewidth=0.5, alpha=0.8)
    axes[0].set_title(f'{CHAPTER} - Original Powerline Signal', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Amplitude', fontsize=12)
    axes[0].set_xlim(0, duration_sec)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Extracted envelope from best frequency
    axes[1].plot(time_axis, best_filtered_norm, color='green', linewidth=0.5, alpha=0.8)
    axes[1].set_title(f'Extracted Normalized Signal from {max_corr_freq/1000:.1f} kHz (±{bandwidth/2000:.1f} kHz) - Correlation: {max_corr_value:.4f}', 
                      fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Normalized Amplitude', fontsize=12)
    axes[1].set_xlim(0, duration_sec)
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: USB signal (normalized)
    axes[2].plot(time_axis, corr_usb_norm, color='blue', linewidth=0.5, alpha=0.8)
    axes[2].set_title(f'{CHAPTER} - USB Signal (Normalized)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Time (seconds)', fontsize=12)
    axes[2].set_ylabel('Normalized Amplitude', fontsize=12)
    axes[2].set_xlim(0, duration_sec)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Audio Playback of Best Frequency Bin

Convert the extracted frequency bin signal to audio for listening.

In [ ]:
# === AUDIO PLAYBACK OF BEST FREQUENCY BIN ===

if 'best_filtered_norm' in globals():
    print(f"🔊 Creating audio from best frequency bin ({max_corr_freq/1000:.1f} kHz ± {bandwidth/2000:.1f} kHz)")
    print(f"   Correlation with USB: {max_corr_value:.4f}")
    
    # Denormalize the signal for audio playback (scale back to reasonable amplitude)
    audio_signal = best_filtered_norm * 0.5  # Scale to prevent clipping
    
    # Display audio player
    print(f"\n🎵 Playing extracted signal from {max_corr_freq/1000:.1f} kHz frequency bin:")
    display(Audio(audio_signal, rate=SAMPLE_RATE))
    
    # Also play the normalized USB signal for comparison
    print(f"\n🎵 For comparison - Original USB signal:")
    usb_audio_normalized = corr_usb_norm * 0.5  # Scale to prevent clipping
    display(Audio(usb_audio_normalized, rate=SAMPLE_RATE))
    
    print(f"\n💡 Listen to both:")
    print(f"   - First audio: Extracted from powerline at {max_corr_freq/1000:.1f} kHz")
    print(f"   - Second audio: Original USB output")
    print(f"   - Correlation: {max_corr_value:.4f} (higher = more similar)")
else:
    print("❌ No frequency bin data available. Please run the correlation analysis cell first.")

## Advanced Signal Processing Pipeline

Complete signal processing pipeline including:
- 60Hz notch filtering
- Bandpass filtering
- Spectrogram-based frequency tracking
- Interpolation and smoothing
- AM modulation experiments
- Downconversion
- Audio playback

In [ ]:
from scipy.io.wavfile import write
from IPython.display import Audio
from scipy import signal
from scipy.signal import hilbert
from scipy.signal import butter, filtfilt, sosfiltfilt, freqz, lfilter, spectrogram
import pandas as pd
from scipy.interpolate import interp1d

# Try to import sounddevice, but continue if not available
try:
    import sounddevice as sd
    AUDIO_AVAILABLE = True
    print("✓ sounddevice loaded - audio playback will be available")
except (OSError, ImportError) as e:
    AUDIO_AVAILABLE = False
    print(f"⚠️ sounddevice not available: {e}")
    print("   Audio playback will be skipped, but all analysis and plots will work.")

def moving_average(signal, window_size):
    """Applies a moving average filter to a 1D signal."""
    return np.convolve(signal, np.ones(window_size)/window_size, mode='valid')

def alternate_merge_loop(arr1, arr2):
    merged_arr = []
    min_len = min(len(arr1), len(arr2))

    for i in range(min_len):
        merged_arr.append(arr1[i])
        merged_arr.append(arr2[i])

    # Append remaining elements from the longer array
    if len(arr1) > min_len:
        merged_arr.extend(arr1[min_len:])
    elif len(arr2) > min_len:
        merged_arr.extend(arr2[min_len:])

    return merged_arr

# === FILE PATHS ===
folder = "May29_Alice"
fsamp = 200000

# Use the data already loaded in the notebook or load fresh
# Check if powerline_data and usb_data already exist from earlier cells
if 'powerline_data' not in locals() or 'usb_data' not in locals():
    usb_path_local = os.path.join(folder, "Chap_1_img.bin")
    powerline_path_local = os.path.join(folder, "Chap_1_real.bin")
    
    # === LOAD SIGNALS ===
    usb_data_orig = np.fromfile(usb_path_local, dtype=np.float32)
    powerline_data_orig = np.fromfile(powerline_path_local, dtype=np.float32)
    
    prenotch_usb_data = usb_data_orig[:2000000]
    prenotch_powerline_data = powerline_data_orig[:2000000]
else:
    # Use data from notebook but limit to first 2M samples
    prenotch_usb_data = usb_data[:2000000]
    prenotch_powerline_data = powerline_data[:2000000]

print(f"Loaded {len(prenotch_powerline_data):,} samples for processing")

# === PLOT SAMPLE SIGNALS ===
plt.figure(figsize=(12, 5))
plt.plot(prenotch_powerline_data[:2000000], label='Powerline (Input)', color='orange')
plt.plot(prenotch_usb_data[:2000000], label='USB (Output)', alpha=0.7, color='blue')
plt.title("Powerline Input vs USB Output - First 2M Samples")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('SamplesOverTime.png')
plt.show()
plt.close()

In [ ]:
# === NOTCH FILTERING (60Hz) ===
f0 = 60  # Freq to be notched
w0 = f0 / (fsamp / 2)  # Normalized Notch Freq
Q = 100  # Quality of the notch (higher the better)
b, a = signal.iirnotch(w0, Q)
powerline_data_notched = signal.filtfilt(b, a, prenotch_powerline_data)
usb_data_notched = signal.filtfilt(b, a, prenotch_usb_data)

print(f"Applied 60Hz notch filter with Q={Q}")

# Compute the Power Spectral Density using Welch's method
frequencies_usb, psd_usb = signal.welch(usb_data_notched, fsamp, nperseg=32768)
frequencies_powerline, psd_powerline = signal.welch(powerline_data_notched, fsamp, nperseg=32768)

# Plot the PSD
plt.figure(figsize=(10, 6))
plt.semilogy(frequencies_usb, psd_usb, label="USB", color='blue')
plt.semilogy(frequencies_powerline, psd_powerline, label="Powerline", color='orange')
plt.title('Power Spectral Density (After 60Hz Notch)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power/Frequency (unit²/Hz)')
plt.grid(True)
plt.legend()
plt.savefig('PowerSpectralDensity.png')
plt.show()
plt.close()

In [ ]:
# === BANDPASS FILTERING ===
order = 4  # Order of the filter

lowcut = 20000  # For USB
highcut = 25000

lowcut_powerline = 18000  # For Powerline
highcut_powerline = 24000

nyq = 0.5 * fsamp
low = lowcut / nyq
high = highcut / nyq

low_powerline = lowcut_powerline / nyq
high_powerline = highcut_powerline / nyq

# Bandpass filter for USB
b_usb, a_usb = butter(order, [low, high], btype='bandpass')
filtered_usb_signal = filtfilt(b_usb, a_usb, usb_data_notched)

frequencies_filtered_usb, psd_filtered_usb = signal.welch(filtered_usb_signal, fsamp, nperseg=32768)

# Bandpass filter for Powerline
b_powerline, a_powerline = butter(order, [low_powerline, high_powerline], btype='bandpass')
filtered_powerline_signal = filtfilt(b_powerline, a_powerline, powerline_data_notched)

frequencies_filtered_powerline, psd_filtered_powerline = signal.welch(filtered_powerline_signal, fsamp, nperseg=32768)

print(f"Applied bandpass filters:")
print(f"  USB: {lowcut}-{highcut} Hz")
print(f"  Powerline: {lowcut_powerline}-{highcut_powerline} Hz")

# === PLOT SAMPLE FILTERED SIGNALS ===
plt.figure(figsize=(12, 5))
plt.plot(filtered_powerline_signal[1000:2000000], label='Powerline (Input)', color='orange')
plt.title("Filtered Powerline Input - Samples 1000-2M")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('FilteredSamplesOverTime.png')
plt.show()
plt.close()

# === PLOT PSD OF FILTERED SIGNALS ===
plt.figure(figsize=(10, 6))
plt.semilogy(frequencies_filtered_usb, psd_filtered_usb, label="Filtered USB", color='blue')
plt.semilogy(frequencies_filtered_powerline, psd_filtered_powerline, label="Filtered Powerline", color='orange')
plt.title('Power Spectral Density (After Bandpass)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power/Frequency (unit²/Hz)')
plt.legend()
plt.grid(True)
plt.savefig('FilteredPowerSpectralDensity.png')
plt.show()
plt.close()

In [ ]:
# === SPECTROGRAM (Freq vs Time) Analysis ===
frequencies, times, Sxx = spectrogram(filtered_powerline_signal, fsamp, nperseg=2048, noverlap=512)
print(f"Spectrogram shape: {Sxx.shape}")

# Highlighting only frequencies close to the max signal strength
df = pd.DataFrame(Sxx)

max_over_time = []
min_over_time = []

for col in df.columns:
    # Find the index of the maximum value in the current column
    max_idx = df[col].idxmax()
    max_value = df[col].max()
    min_value = df[col].min()

    # Create a boolean mask for the maximum value and its neighbors
    mask = np.zeros(len(df), dtype=bool)
    mask[max_idx] = True
    avg_val = 0
    j = 5  # Specifies the range of frequencies around the max to retain for plotting
    
    if max_idx > j:
        for v in range(j):
            mask[max_idx - v] = True
            avg_val = avg_val + frequencies[max_idx - v]
    if max_idx < len(df) - j:
        for v in range(j):
            mask[max_idx + v] = True
            avg_val = avg_val + frequencies[max_idx + v]
    
    avg_val = avg_val / (2 * j)
    
    # Apply the mask to the column
    df[col] = df[col].where(mask, 0)
    
    # Note down the frequencies where max is observed and normalize
    max_over_time.append((frequencies[max_idx + 2*j] - lowcut_powerline) / 500)
    min_over_time.append((frequencies[max_idx - 2*j] - lowcut_powerline) / 500)

print(f"Extracted {len(max_over_time)} frequency track points")

# Plot spectrogram
plt.figure(figsize=(10, 6))
plt.pcolormesh(times, frequencies, 10 * np.log10(df), shading='gouraud')
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [s]')
plt.title('Spectrogram (Frequency Tracking)')
plt.colorbar(label='Power/Frequency (dB/Hz)')
plt.ylim([lowcut_powerline-1000, highcut_powerline+1000])
plt.savefig('FreqVsTimePlot.png')
plt.show()
plt.close()

# === PLOT MAX VALUES OF SPECTROGRAM ===
plt.figure(figsize=(12, 5))
plt.plot(max_over_time[:2000000], label='Max_Over_Time', color='orange')
plt.title("Max Frequency Over Time - First 2M Samples")
plt.xlabel("Sample Index")
plt.ylabel("Normalized Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('MaxOverTime.png')
plt.show()
plt.close()

In [ ]:
# === INTERPOLATION ===
# Dilate the signal over time to match the original audio
original_x = np.linspace(0, 1, len(max_over_time))
new_x = np.linspace(0, 1, 1000000)
f = interp1d(original_x, max_over_time, kind='linear')
interpolate_max_over_time = f(new_x)

original_x1 = np.linspace(0, 1, len(min_over_time))
new_x1 = np.linspace(0, 1, 1000000)
f = interp1d(original_x1, min_over_time, kind='linear')
interpolate_min_over_time = f(new_x1)

new_interpolated_over_time = alternate_merge_loop(interpolate_max_over_time, interpolate_min_over_time)

print(f"Interpolated signal length: {len(new_interpolated_over_time):,}")

# === PLOT INTERPOLATED MAX VALUES ===
plt.figure(figsize=(10, 6))
plt.plot(new_interpolated_over_time[:2000000], label='Interpolated', color='orange')
plt.title("Interpolated Signal")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Interpolated.png')
plt.show()
plt.close()

In [ ]:
# === SMOOTHING ===
# Smoothen the interpolated signal
smoothed_signal = moving_average(new_interpolated_over_time, window_size=2)

print(f"Smoothed signal length: {len(smoothed_signal):,}")

# === PLOT SMOOTHENED SIGNAL ===
plt.figure(figsize=(10, 6))
plt.plot(smoothed_signal[:2000000], label='Smoothened', color='orange')
plt.title("Smoothened Signal")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('Smoothened.png')
plt.show()
plt.close()

In [ ]:
# === AM MODULATION ===
tot_time_max = len(interpolate_max_over_time) / fsamp
time_interpolate_max = np.arange(0, tot_time_max, 1/fsamp)
fc = 10000
am_signal = abs(interpolate_max_over_time) * np.cos(2 * np.pi * fc * time_interpolate_max)

print(f"AM signal length: {len(am_signal):,}")

# === PLOT SAMPLE AM MODULATED MAX SIGNALS ===
plt.figure(figsize=(12, 5))
plt.plot(am_signal[:2000000], label='AM Modulated Max', color='green')
plt.title("AM Modulated Signal - First 2M Samples")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('AMModulatedSamplesOverTime.png')
plt.show()
plt.close()

In [ ]:
# === DOWNCONVERTING FILTERED SIGNALS ===
downconvert_freq = lowcut
downconvert_freq_powerline = lowcut_powerline
tot_time_usb = len(filtered_usb_signal) / fsamp
tot_time_powerline = len(filtered_powerline_signal) / fsamp

time_usb = np.arange(0, tot_time_usb, 1/fsamp)
time_powerline = np.arange(0, tot_time_powerline, 1/fsamp)

local_oscillator_usb = np.exp(-1j * 2 * np.pi * downconvert_freq * time_usb)
local_oscillator_powerline = np.exp(-1j * 2 * np.pi * downconvert_freq_powerline * time_powerline)

# Mix the input signal with the local oscillator
down_usb = filtered_usb_signal * local_oscillator_usb
frequencies_down_usb, psd_down_usb = signal.welch(down_usb, fsamp, nperseg=32768)

down_powerline = filtered_powerline_signal * local_oscillator_powerline
frequencies_down_powerline, psd_down_powerline = signal.welch(down_powerline, fsamp, nperseg=32768)

scaling = abs(down_usb.max()) / abs(interpolate_max_over_time.max())
print(f"Scaling value: {scaling}")

frequencies_interpol_max, psd_interpol_max = signal.welch(interpolate_max_over_time[:200000]*scaling, fsamp, nperseg=32768)

# === PLOT PSD OF DOWNCONVERTED SIGNALS ===
plt.figure(figsize=(10, 6))
plt.semilogy(frequencies_down_usb, psd_down_usb, label="Down USB", color='blue')
plt.semilogy(frequencies_down_powerline, psd_down_powerline, label="Down Powerline", color='orange')
plt.semilogy(frequencies_interpol_max, psd_interpol_max, label="Interpol Max", color='green')
plt.title('Power Spectral Density (Downconverted)')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power/Frequency (unit²/Hz)')
plt.legend()
plt.grid(True)
plt.savefig('DownConvertedPowerSpectralDensity.png')
plt.show()
plt.close()

# === PLOT SAMPLE DOWNCONVERTED SIGNALS ===
plt.figure(figsize=(12, 5))
plt.plot(down_usb[:2000000], label='USB', color='orange')
plt.title("Downconverted USB - First 2M Samples")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('DownconvertedUSBSamplesOverTime.png')
plt.show()
plt.close()

In [ ]:
# === AUDIO PLAYBACK ===
# Note: Audio playback requires sounddevice which may not work in all environments

print("=" * 70)
print("AUDIO PLAYBACK SECTION")
print("=" * 70)

if not AUDIO_AVAILABLE:
    print("\n⚠️ Audio playback is not available (sounddevice/PortAudio not installed)")
    print("All signal processing and visualization completed successfully!")
    print("You can use IPython.display.Audio as an alternative for playback in notebooks.")
    print("=" * 70)
else:
    try:
        # === PLAYING DOWNCONVERTED USB AUDIO ===
        print("\n🔊 Playing DownConverted USB Audio...")
        sd.play(np.abs(down_usb[100:2000000])*10, fsamp)
        sd.wait()
        print("✓ Finished playing USB audio")
        
        # === PLAYING INTERPOLATED AUDIO ===
        print(f"\n🔊 Playing Interpolated Audio (length: {len(new_interpolated_over_time):,})...")
        sd.play(np.abs(new_interpolated_over_time), fsamp)
        sd.wait()
        print("✓ Finished playing interpolated audio")
        
        # === PLAYING SMOOTHENED AUDIO ===
        print(f"\n🔊 Playing Smoothened Audio (length: {len(smoothed_signal):,})...")
        sd.play(np.abs(smoothed_signal)/np.abs(smoothed_signal.max()), fsamp)
        sd.wait()
        print("✓ Finished playing smoothened audio")
        
        print("\n" + "=" * 70)
        print("All audio playback completed!")
        print("=" * 70)
        
    except Exception as e:
        print(f"\n⚠️ Audio playback error: {e}")
        print("This is normal if sounddevice is not available or configured.")
        print("You can still view all the generated plots and analysis results.")

## Spectrogram of AM Modulated Signal

Generate spectrogram visualization of the AM modulated signal.

In [ ]:
# === SPECTROGRAM OF AM MODULATED SIGNAL ===
import librosa.display
from scipy import signal as sig

if 'am_signal' in globals():
    print(f"🎼 Generating Spectrogram for AM Modulated Signal")
    print(f"   Signal length: {len(am_signal):,} samples")
    print(f"   Sample rate: {fsamp} Hz")
    print(f"   Duration: {len(am_signal)/fsamp:.2f} seconds")
    print(f"   Carrier frequency: {fc} Hz")
    
    # Create spectrogram
    fig, ax = plt.subplots(1, 1, figsize=(16, 6))
    
    # Use scipy.signal.spectrogram for detailed analysis
    frequencies_spec, times_spec, Sxx = sig.spectrogram(np.array(new_interpolated_over_time[:2000000]), fsamp, nperseg=4096)
    
    # Limit to 0-20 kHz range
    freq_mask = (frequencies_spec >= 0) & (frequencies_spec <= 20000)
    
    img = ax.pcolormesh(times_spec, frequencies_spec[freq_mask], 
                        10 * np.log10(Sxx[freq_mask, :] + 1e-10), 
                        shading='gouraud', cmap='viridis')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title(f'Interpolated Signal Spectrogram', 
                 fontsize=12, fontweight='bold')
    ax.set_ylim([0, 10000])
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    
    plt.tight_layout()
    
    plt.show()
    
    print("\n✅ Spectrogram generated successfully!")
    
    
    
else:
    print("❌ AM signal not found. Please run the AM modulation cell first!")

## MP3 Reference Spectrogram

Load and display the spectrogram of the original MP3 file for the same chapter and duration.

In [ ]:
# === SPECTROGRAM OF MP3 REFERENCE ===
import librosa
from scipy import signal as sig
import re

# Force load MP3 reference (use provided stepped tones sweep)
mp3_path = '<REPO_ROOT>/Audio Files/stepped_tones_sweep.mp3'
chapter_num = 1  # First chapter (metadata only)
mp3_duration = 10.0  # 5 seconds

try:
    
    if os.path.exists(mp3_path):
        print(f"🎵 Loading MP3 Reference: {os.path.basename(mp3_path)}")
        print(f"   Chapter: {chapter_num}")
        print(f"   Duration: {mp3_duration} seconds")
        
        # Load MP3 audio with librosa
        mp3_audio, mp3_sr = librosa.load(mp3_path, sr=None, mono=True)
        
        # Extract first 5 seconds
        mp3_samples = int(mp3_duration * mp3_sr)
        mp3_extract = mp3_audio[:mp3_samples]
        
        print(f"   MP3 sample rate: {mp3_sr} Hz")
        print(f"   MP3 actual duration: {len(mp3_extract)/mp3_sr:.2f} seconds")
        print(f"   MP3 samples: {len(mp3_extract):,}")
        
        # Create figure with 2 subplots: time domain on top, spectrogram below
        fig, (ax_time, ax_spec) = plt.subplots(2, 1, figsize=(16, 10), 
                                                gridspec_kw={'height_ratios': [1, 2]})
        
        # Plot 1: Time domain waveform
        time_axis = np.arange(len(mp3_extract)) / mp3_sr
        ax_time.plot(time_axis, mp3_extract, linewidth=0.5, color='blue')
        ax_time.set_ylabel('Amplitude')
        ax_time.set_xlabel('Time (s)')
        ax_time.set_title(f'MP3 Time Domain - Chapter {chapter_num}, First {mp3_duration}s', 
                          fontsize=12, fontweight='bold')
        ax_time.grid(True, alpha=0.3)
        ax_time.set_xlim([0, mp3_duration])
        
        # Plot 2: Spectrogram
        # Use scipy.signal.spectrogram for detailed analysis
        frequencies_spec, times_spec, Sxx = sig.spectrogram(mp3_extract, mp3_sr, nperseg=512)
        
        # Limit to 0-20 kHz range
        freq_mask = (frequencies_spec >= 0) & (frequencies_spec <= 20000)
        
        img = ax_spec.pcolormesh(times_spec, frequencies_spec[freq_mask], 
                            10 * np.log10(Sxx[freq_mask, :] + 1e-10), 
                            shading='gouraud', cmap='viridis')
        ax_spec.set_ylabel('Frequency (Hz)')
        ax_spec.set_xlabel('Time (s)')
        ax_spec.set_title(f'MP3 Spectrogram - Chapter {chapter_num}, First {mp3_duration}s (0-20 kHz)', 
                     fontsize=12, fontweight='bold')
        ax_spec.set_ylim([0, 10000])
        #fig.colorbar(img, ax=ax_spec, format='%+2.0f dB')
        
        plt.tight_layout()
        plt.savefig('MP3_Reference_Spectrogram.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\n✅ MP3 spectrogram generated successfully!")
        print(f"   - Saved as 'MP3_Reference_Spectrogram.png'")
        print(f"   - Frequency range: 0-20 kHz")
        print(f"   - Duration: {mp3_duration}s")
        
    else:
        print(f"❌ MP3 file not found: {mp3_path}")

except Exception as e:
    print(f"❌ Error loading MP3: {e}")
    import traceback
    traceback.print_exc()

# Demo Files Plots

## Time Domain Comparison: Binary vs MP3 Files

In [ ]:
# === TIME DOMAIN vs FREQUENCY DOMAIN COMPARISON: BINARY vs MP3 FILES ===
import os, glob, numpy as np, matplotlib.pyplot as plt
from scipy.signal import spectrogram, welch
try:
    import librosa
    librosa_available = True
except ImportError:
    librosa_available = False
    print("Warning: librosa not available, cannot load MP3 files")

DEMO_DIR = '/fs/scratch/<allocation>/nov_demo'
AUDIO_DIR = '/fs/scratch/<allocation>/Audio Files'

# Get binary and MP3 files
real_files = sorted(glob.glob(os.path.join(DEMO_DIR, '*_real.bin')))
mp3_files = sorted(glob.glob(os.path.join(AUDIO_DIR, '*.mp3')))

print(f'Found {len(real_files)} binary files and {len(mp3_files)} MP3 files')

if librosa_available and len(real_files) > 0 and len(mp3_files) > 0:
    # Create mapping between binary and MP3 files based on improved name matching
    file_pairs = []
    for real_file in real_files:
        real_basename = os.path.basename(real_file)
        # Extract base name without _real.bin suffix
        if real_basename.endswith('_real.bin'):
            real_name = real_basename.replace('_real.bin', '')
        else:
            real_name = real_basename.replace('.bin', '')
        
        # Look for matching MP3 file with exact name match
        matching_mp3 = None
        for mp3_file in mp3_files:
            mp3_basename = os.path.basename(mp3_file)
            mp3_name = mp3_basename.replace('.mp3', '')
            
            # Direct match or close variants
            if real_name == mp3_name or real_name in mp3_name or mp3_name in real_name:
                matching_mp3 = mp3_file
                break
        
        if matching_mp3:
            file_pairs.append((real_file, matching_mp3))
            print(f'Paired: {os.path.basename(real_file)} <-> {os.path.basename(matching_mp3)}')
        else:
            print(f'No matching MP3 found for: {os.path.basename(real_file)}')
    
    # Plot comparisons for each pair
    for idx, (real_file, mp3_file) in enumerate(file_pairs):
        real_name = os.path.basename(real_file)
        mp3_name = os.path.basename(mp3_file)
        
        print(f'\n--- Comparing {real_name} (time) vs {mp3_name} (frequency) ---')
        
        try:
            # Load binary file
            real_data = np.fromfile(real_file, dtype=np.float32)
            if len(real_data) == 0:
                print(f'No data in {real_name}, skipping')
                continue
            
            # Load MP3 file
            mp3_data, mp3_sr = librosa.load(mp3_file, sr=None, mono=True)
            
            # Use sample rate from globals or MP3 file
            fs_real = globals().get('SAMPLE_RATE', 200000)
            fs_mp3 = mp3_sr
            
            # Create time axis for binary file
            t_real = np.arange(len(real_data)) / float(fs_real)
            
            # Determine common time duration for both plots
            max_time = min(len(real_data)/fs_real, len(mp3_data)/fs_mp3)  # Max 10 seconds or file duration
            
            # Limit data length based on common time duration
            max_samples_real = min(len(real_data), int(max_time * fs_real))
            max_samples_mp3 = min(len(mp3_data), int(max_time * fs_mp3))
            
            show_real = real_data[:max_samples_real]
            show_mp3 = mp3_data[:max_samples_mp3]
            t_real_show = t_real[:max_samples_real]
            
            # Create comparison plot: Time domain (binary) vs Frequency domain (MP3)
            fig, axes = plt.subplots(2, 1, figsize=(14, 10))
            
            # 1. Binary file - Time Domain
            axes[0].plot(t_real_show, show_real, 'b-', linewidth=0.8, alpha=0.8)
            axes[0].set_title(f'{real_name} - Time Domain Signal (Binary, {fs_real} Hz)')
            axes[0].set_xlabel('Time (s)')
            axes[0].set_ylabel('Amplitude')
            axes[0].grid(True, alpha=0.3)
            axes[0].set_xlim(0, max_time)  # Set common time axis
            
            # 2. MP3 file - Spectrogram (Frequency Domain)
            f_spec, t_spec, Sxx_mp3 = spectrogram(show_mp3, fs=fs_mp3, nperseg=1024, noverlap=512)
            im = axes[1].pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx_mp3 + 1e-12), 
                                   shading='gouraud', cmap='viridis')
            axes[1].set_title(f'{mp3_name} - Spectrogram (MP3, {fs_mp3} Hz)')
            axes[1].set_xlabel('Time (s)')
            axes[1].set_ylabel('Frequency (Hz)')
            # Focus on relevant frequency range for powerline analysis
            axes[1].set_ylim(0, min(25000, fs_mp3/2))
            axes[1].set_xlim(0, max_time)  # Set common time axis
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f'Error processing pair {real_name}/{mp3_name}: {e}')
            continue

else:
    if not librosa_available:
        print("Cannot perform comparison - librosa is required to load MP3 files")
    else:
        print("No files found to compare")

print('\nDone with time/frequency domain comparison')

## Power Spectral Density - All Real Files

In [ ]:
# === PSD ANALYSIS OF ALL REAL FILES ===
import os, glob, numpy as np, matplotlib.pyplot as plt
from scipy.signal import welch

DEMO_DIR = '/fs/scratch/<allocation>/nov_demo'

# Get all real files
real_files = sorted(glob.glob(os.path.join(DEMO_DIR, '*_real.bin')))
print(f'Found {len(real_files)} real files for PSD analysis')

if len(real_files) > 0:
    # Use default sample rate
    fs = globals().get('SAMPLE_RATE', 200000)
    
    plt.figure(figsize=(16, 8))
    
    # Color palette for different files
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
    
    for idx, real_file in enumerate(real_files):
        real_name = os.path.basename(real_file)
        print(f'Processing {real_name}...')
        
        try:
            # Load real file data
            data = np.fromfile(real_file, dtype=np.float32)
            if len(data) == 0:
                print(f'  No data in {real_name}, skipping')
                continue
            
            # Compute PSD using Welch's method
            f_psd, psd = welch(data, fs=fs, nperseg=2048, noverlap=1024)
            
            # Convert to dB
            psd_db = 10 * np.log10(psd + 1e-12)
            
            # Get color for this file
            color = colors[idx % len(colors)]
            
            # Plot PSD
            plt.plot(f_psd, psd_db, color=color, label=real_name, alpha=0.8, linewidth=1.5)
            
        except Exception as e:
            print(f'  Error processing {real_name}: {e}')
            continue
    
    # Customize plot
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Power Spectral Density (dB/Hz)')
    plt.title('Power Spectral Density - All Real Files')
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Set frequency range (0 to 80 kHz for powerline analysis)
    plt.xlim(0, min(80000, fs/2))
    
    

    
    plt.tight_layout()
    plt.show()
    
    print(f'\nPSD analysis complete for {len(real_files)} files')
    print(f'Sample rate used: {fs} Hz')
    print(f'Frequency range: 0 - {min(25000, fs/2)} Hz')
    
else:
    print("No real files found for PSD analysis")

print('\nDone with PSD analysis')

## High-Contrast Spectrograms - 20 kHz Band (15-25 kHz)

In [ ]:
# === HIGH-CONTRAST SPECTROGRAMS - 20 kHz BAND ===
import os, glob, numpy as np, matplotlib.pyplot as plt
from scipy.signal import spectrogram
import matplotlib.colors as colors

DEMO_DIR = '/fs/scratch/<allocation>/nov_demo'

# Get all real files
real_files = sorted(glob.glob(os.path.join(DEMO_DIR, '*_real.bin')))
print(f'Found {len(real_files)} real files for spectrogram analysis')

if len(real_files) > 0:
    # Use default sample rate
    fs = globals().get('SAMPLE_RATE', 200000)
    
    # Target frequency band: 20 kHz ± 5 kHz (15-25 kHz)
    freq_min = 0.1  # Hz
    freq_max = 100  # Hz
    
    # Analysis time window parameters
    start_time = 0.0    # Start time in seconds (change this to analyze different parts)
    duration = 30.0     # Duration in seconds to analyze
    
    print(f'Creating high-contrast spectrograms for {freq_min}-{freq_max} Hz band')
    print(f'Time window: {start_time:.1f}s to {start_time + duration:.1f}s')
    
    for idx, real_file in enumerate(real_files):
        real_name = os.path.basename(real_file)
        print(f'\nProcessing {real_name}...')
        
        try:
            # Load real file data
            data = np.fromfile(real_file, dtype=np.float32)
            if len(data) == 0:
                print(f'  No data in {real_name}, skipping')
                continue
            
            # Calculate sample indices for the time window
            start_sample = int(start_time * fs)
            end_sample = min(len(data), int((start_time + duration) * fs))
            
            # Check if start_sample is valid
            if start_sample >= len(data):
                print(f'  Start time {start_time}s exceeds file duration {len(data)/fs:.1f}s, skipping')
                continue
                
            # Extract the specified time window
            data_show = data[start_sample:end_sample]
            actual_duration = len(data_show) / fs
            
            # Compute spectrogram with higher resolution
            f, t, Sxx = spectrogram(data_show, fs=fs, nperseg=4096, noverlap=3072)
            
            # Convert to dB
            Sxx_db = 10 * np.log10(Sxx + 1e-12)
            
            # Create high-contrast black and white plot
            plt.figure(figsize=(14, 6))
            
            # Apply aggressive contrast enhancement
            # Calculate percentiles for dynamic range
            p_low = np.percentile(Sxx_db, 20)   # 10th percentile
            p_high = np.percentile(Sxx_db, 90)  # 95th percentile
            
            # Clip and normalize for high contrast
            Sxx_contrast = np.clip(Sxx_db, p_low, p_high)
            Sxx_contrast = (Sxx_contrast - p_low) / (p_high - p_low)
            
            # Create binary-like appearance by applying threshold
            threshold = 0.3  # Adjust for more/less contrast
            Sxx_binary = np.where(Sxx_contrast > threshold, 1.0, 0.0)
            
            # Plot with pure black and white colormap
            im = plt.pcolormesh(t, f, Sxx_binary, shading='gouraud', cmap='gray', vmin=0, vmax=1)
            
            # Focus on the 20 kHz band
            plt.ylim(freq_min, freq_max)
            
            # Customize appearance for high contrast
            plt.xlabel('Time (s)', fontsize=12, fontweight='bold')
            plt.ylabel('Frequency (Hz)', fontsize=12, fontweight='bold')
            plt.title(f'{real_name} - High-Contrast Spectrogram ({freq_min/1000:.0f}-{freq_max/1000:.0f} kHz)', 
                     fontsize=14, fontweight='bold')
            
            # Add frequency markers
            freq_markers = [16000, 18000, 20000, 22000, 24000, 26000, 28000, 30000]
            for freq in freq_markers:
                if freq_min <= freq <= freq_max:
                    plt.axhline(y=freq, color='red', linestyle='-', alpha=0.7, linewidth=1)
                    plt.text(plt.xlim()[1]*0.02, freq, f'{freq/1000:.0f}k', 
                            va='center', ha='left', color='red', fontweight='bold', fontsize=10)
            
            # Highlight 20 kHz center frequency
            plt.axhline(y=20000, color='red', linestyle='-', alpha=1.0, linewidth=2)
            
            # Set black background for axes
            plt.gca().set_facecolor('black')
            
            # Remove colorbar for cleaner look
            plt.tight_layout()
            plt.show()
            
            print(f'  Processed: {len(data_show)/fs:.1f}s of data')
            print(f'  Frequency resolution: {f[1]-f[0]:.1f} Hz')
            print(f'  Time resolution: {t[1]-t[0]:.3f} s')
            
        except Exception as e:
            print(f'  Error processing {real_name}: {e}')
            continue
    
    print(f'\nHigh-contrast spectrogram analysis complete')
    print(f'Target band: {freq_min} - {freq_max} Hz')
    print(f'Sample rate: {fs} Hz')
    
else:
    print("No real files found for spectrogram analysis")

print('\nDone with high-contrast spectrogram analysis')